In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.2 SFT

> 🎯 **Goal:** Teach the base model the *job of answering* by training on worked
> examples of prompt then response, while grading only the response.

**Why this matters.** This is the first and most important post-training step. SFT
(supervised fine-tuning) is what takes the well-read-but-clueless base model from
l7.1 and turns it into something that actually replies. Nearly every instruction
model you have used started life as a base model that went through SFT.

**The intuition.** Imagine teaching a student by showing worked examples: here is a
problem, here is the correct solution. But you grade them only on the *solution*
part, never on copying the problem back. If you graded them on rewriting the
question, they would learn to be great at echoing questions, which is useless. By
grading only the answer, you push them to learn the mapping from prompt to response.

**The mechanics.** You assemble a dataset of `(prompt, response)` pairs, for example
('Q: hi A:', ' hello'). You concatenate them and train with the same next-token
objective as pretraining, with one decisive change: you **mask the loss on the
prompt tokens**. Masking means setting those target positions to a sentinel value
(`-100`, the value `cross_entropy` ignores) so they contribute nothing to the loss
and nothing to the gradient. The model still *reads* the prompt (it needs the
context to predict well), but it is only *graded* on producing the response. That
single masking trick is what converts a continuation engine into an
instruction-follower, and it is identical at every scale from this toy to a frontier
model.

In [ ]:
from torch.nn import functional as F
from pocketlm import CharTokenizer, PocketLM, PocketLMConfig, init_weights

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
prompt, response = "Q: hi A:", " hello"
ids = torch.tensor([tok.encode(prompt + response)])
x = ids[:, :-1]
y = ids[:, 1:].clone()
prompt_len = len(tok.encode(prompt))
y[:, :prompt_len - 1] = -100   # ignore prompt tokens in the loss

torch.manual_seed(0)
model = init_weights(PocketLM(PocketLMConfig(vocab_size=tok.vocab_size, block_size=64,
                                             n_layer=2, n_head=2, n_embd=64)))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
first = None
for _ in range(150 if os.environ.get("POCKETLM_CI") else 300):
    logits, _ = model(x)
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1),
                           ignore_index=-100)
    if first is None:
        first = loss.item()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("response loss:", round(first, 3), "->", round(loss.item(), 3))

**What just happened.** The printed loss fell sharply, from its starting value
toward near zero. With a single pair the model simply memorizes the response, which
is exactly what we want to see for a sanity check: the masked objective is
learnable and the gradient flows only through the response tokens. On real data you
would use many thousands of diverse pairs so the model learns the *general* skill
of answering rather than one canned reply.

⚠️ **Common pitfalls**
- Off-by-one in the mask. Because targets are shifted by one (`y = ids[:, 1:]`), the
  prompt mask uses `prompt_len - 1`. Mask the wrong boundary and you either grade
  the model on the prompt or skip part of the response.
- Training on one example and declaring victory. Loss near zero here means
  memorization, not generalization. Real SFT needs diverse, high-quality pairs.
- Forgetting `ignore_index=-100`. Without it the masked positions still count and
  the model is rewarded for echoing the prompt.

🏋️ **Try it yourself.** Add a second pair, for example ('Q: bye A:', ' goodbye'),
stack both into the batch, and rerun. Watch how the loss behaves when the model has
to learn two mappings at once instead of memorizing one. Then deliberately remove
the masking line and observe the model start to waste capacity echoing prompts.

In [ ]:
assert loss.item() < first